### 使用val + train 一部分近期3年数据，看看绩效

In [1]:
import pandas  as pd
import numpy as np
import pdb, os, datetime, itertools, time, hashlib
from lumina.genetic.util import create_id
from dotenv import load_dotenv

load_dotenv()
from lib.flp001 import *

In [2]:
method = 'ricso2'
task_id = '113001'
instruments = 'rbb'
period = 5
category='p'

In [3]:
draft_data = load_data3(method=method, task_id=task_id, instruments=instruments, 
          period=period, category='p',filename='choose.csv')
draft_data.sort_values(by=['ann_sharpe'],ascending=False)[['formula','ic_mean','ann_sharpe','calmar','pl_ratio']]

,formula,ic_mean,ann_sharpe,calmar,pl_ratio
2,SIGLOG2ABS('oi034_5_10_1'),-0.1250,3.58,8.68,None
0,SIGLOG10ABS(SIGLOG2ABS(SIGLOG2ABS('oi034_5_10_...,-0.1181,3.53,8.35,None
3,"MPERCENT(120,MMinDiff(90,'tv018_2_3_1'))",-0.0766,3.13,5.09,None
4,"MPERCENT(120,MMinDiff(90,MMinDiff(90,'tv018_2_...",-0.0743,3.11,4.95,None
1,"SIGLOG10ABS(SIGLOG2ABS(SIGLOG10ABS(EMA(120,MPE...",0.2436,2.91,4.50,None


In [4]:
quantile = 0.3
base_ic = draft_data['ic_mean'].abs().sort_values().quantile(quantile).round(3)
base_sharpe = draft_data['ann_sharpe'].sort_values().quantile(quantile).round(3)
base_calmar = draft_data['calmar'].sort_values().quantile(quantile).round(3)
base_pl_ratio = draft_data['pl_ratio'].sort_values().quantile(0.1).round(3)
base_result = {
"abs_ic": base_ic,
"ann_sharpe": base_sharpe,
"calmar": base_calmar,
"pl_ratio ":base_pl_ratio
}
pd.DataFrame([base_result])

,abs_ic,ann_sharpe,calmar,pl_ratio
0,0.085,3.114,4.978,NaN


In [5]:
draft_data

,factor_id,formula,category,direction,source,ic_mean,ann_sharpe,calmar,max_dd,avg_ret,total_ret,win_rate,pl_ratio,turnover,factor_ac,plot
0,10410898,SIGLOG10ABS(SIGLOG2ABS(SIGLOG2ABS('oi034_5_10_...,p,-1,20260401,-0.1181,3.53,8.35,-5.31,0.23,350.07,None,None,0.4027,0.4068,./records/ricso2/rbb/rulex/113001/nxt1_ret_5h/...
1,10805732,"SIGLOG10ABS(SIGLOG2ABS(SIGLOG10ABS(EMA(120,MPE...",p,1,20260401,0.2436,2.91,4.50,-8.26,0.20,264.67,None,None,0.3726,0.9116,./records/ricso2/rbb/rulex/113001/nxt1_ret_5h/...
2,10094815,SIGLOG2ABS('oi034_5_10_1'),p,-1,20260401,-0.1250,3.58,8.68,-5.21,0.23,361.57,None,None,0.4075,0.4375,./records/ricso2/rbb/rulex/113001/nxt1_ret_5h/...
3,10306984,"MPERCENT(120,MMinDiff(90,'tv018_2_3_1'))",p,-1,20260428,-0.0766,3.13,5.09,-6.24,0.17,209.87,None,None,0.4247,0.0737,./records/ricso2/rbb/rulex/113001/nxt1_ret_5h/...
4,10330046,"MPERCENT(120,MMinDiff(90,MMinDiff(90,'tv018_2_...",p,-1,20260428,-0.0743,3.11,4.95,-6.38,0.17,207.97,None,None,0.4237,0.0772,./records/ricso2/rbb/rulex/113001/nxt1_ret_5h/...


In [6]:
draft_data1 = draft_data[
    # (draft_data['pl_ratio']>= base_pl_ratio) & 
    (draft_data['ann_sharpe'] >= base_sharpe) & (
        draft_data['calmar'] >= base_calmar) &(
        draft_data['ic_mean'].abs() >= base_ic
        )]

draft_data1

,factor_id,formula,category,direction,source,ic_mean,ann_sharpe,calmar,max_dd,avg_ret,total_ret,win_rate,pl_ratio,turnover,factor_ac,plot
0,10410898,SIGLOG10ABS(SIGLOG2ABS(SIGLOG2ABS('oi034_5_10_...,p,-1,20260401,-0.1181,3.53,8.35,-5.31,0.23,350.07,None,None,0.4027,0.4068,./records/ricso2/rbb/rulex/113001/nxt1_ret_5h/...
2,10094815,SIGLOG2ABS('oi034_5_10_1'),p,-1,20260401,-0.1250,3.58,8.68,-5.21,0.23,361.57,None,None,0.4075,0.4375,./records/ricso2/rbb/rulex/113001/nxt1_ret_5h/...


In [7]:
draft_data1['plot'] = draft_data1['plot'].apply(make_clickable)

In [8]:
# draft_data.to_csv("rbb1.csv",encoding="UTF-8")
# draft_data.to_csv("pro_rbb.csv",encoding="UTF-8")
draft_data1.head()

,factor_id,formula,category,direction,source,ic_mean,ann_sharpe,calmar,max_dd,avg_ret,total_ret,win_rate,pl_ratio,turnover,factor_ac,plot
0,10410898,SIGLOG10ABS(SIGLOG2ABS(SIGLOG2ABS('oi034_5_10_...,p,-1,20260401,-0.1181,3.53,8.35,-5.31,0.23,350.07,None,None,0.4027,0.4068,"<a target=""_blank"" href=""./records/ricso2/rbb/..."
2,10094815,SIGLOG2ABS('oi034_5_10_1'),p,-1,20260401,-0.1250,3.58,8.68,-5.21,0.23,361.57,None,None,0.4075,0.4375,"<a target=""_blank"" href=""./records/ricso2/rbb/..."


In [9]:
to_html(draft_data1[['factor_id','direction','formula','source','plot']])

url,direction,formula,source,plot,factor_id
10410898,-1,SIGLOG10ABS(SIGLOG2ABS(SIGLOG2ABS('oi034_5_10_1'))),20260401,./records/ricso2/rbb/rulex/113001/nxt1_ret_5h/recent/10410898/comparison_plot.png,10410898
10094815,-1,SIGLOG2ABS('oi034_5_10_1'),20260401,./records/ricso2/rbb/rulex/113001/nxt1_ret_5h/recent/10094815/comparison_plot.png,10094815
